# AI-Generated Text Detection - Model Comparison & Fine-tuning Pipeline

This notebook provides:
1. Comparison of AI-text detection models
2. Fine-tuning workflow for custom models
3. Precision/recall analysis and threshold optimization
4. Per-category breakdowns for different AI models

In [ ]:
# AI Text Detection Analysis
import os
import json
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch

from dotenv import load_dotenv
load_dotenv()

print("Libraries loaded successfully!")

In [ ]:
# AI Text Detection Models to Compare
AI_TEXT_MODELS = [
    {
        "name": "RoBERTa OpenAI Detector",
        "model_id": "roberta-base-openai-detector",
        "description": "Fine-tuned to detect GPT-2/GPT-3 generated text"
    },
    {
        "name": "GPTNeo-2.7B Detector",
        "model_id": "EleutherAI/gpt-neo-2.7B",
        "description": "For generating synthetic text samples"
    },
    {
        "name": "BERT GPT-2 Detector",
        "model_id": "基地/GPT-Detect",
        "description": "Alternative GPT detection model"
    }
]

# Test samples with different AI models
AI_TEST_SAMPLES = [
    {
        "text": "In conclusion, the research demonstrates significant improvements in the proposed methodology.",
        "label": "ai",
        "source": "academic_ai"
    },
    {
        "text": "Once upon a time, there was a magical kingdom where dreams came true every night.",
        "label": "ai",
        "source": "creative_ai"
    },
    {
        "text": "I think this article makes some good points about the current economic situation.",
        "label": "human",
        "source": "human_written"
    },
    {
        "text": "The implementation of machine learning algorithms requires careful consideration of various factors.",
        "label": "ai",
        "source": "technical_ai"
    },
    {
        "text": "Just wanted to share my thoughts on today's meeting - it was really productive!",
        "label": "human",
        "source": "casual_human"
    },
    {
        "text": "Therefore, it can be concluded that the hypothesis was correct based on the evidence presented.",
        "label": "ai",
        "source": "formal_ai"
    }
]

print(f"Loaded {len(AI_TEST_SAMPLES)} test samples")
print(f"Comparing {len(AI_TEXT_MODELS)} AI detection models")

In [ ]:
# Benchmark AI detection models
ai_results = []

for model_info in AI_TEXT_MODELS:
    print(f"\nTesting {model_info['name']}...")
    
    try:
        # Skip generator models for detection
        if "GPTNeo" in model_info['model_id']:
            print("  Skipped (generator model)")
            continue
            
        hf_token = os.getenv("HF_TOKEN")
        pipeline_kwargs = {"model": model_info["model_id"]}
        if hf_token:
            pipeline_kwargs["token"] = hf_token
        
        pipe = pipeline("text-classification", **pipeline_kwargs, truncation=True, max_length=512)
        
        predictions = []
        inference_times = []
        
        for sample in AI_TEST_SAMPLES:
            start = time.time()
            result = pipe(sample["text"][:500])
            inference_times.append(time.time() - start)
            
            if isinstance(result, list) and len(result) > 0:
                top = result[0]
                label = str(top.get("label", "")).lower()
                score = float(top.get("score", 0.5))
                
                # Normalize to ai/human
                is_ai = "ai" in label or "generated" in label or "fake" in label or label == "1"
                predictions.append({
                    "predicted": is_ai,
                    "confidence": score,
                    "raw_label": label
                })
            else:
                predictions.append({"predicted": False, "confidence": 0.5, "raw_label": "unknown"})
        
        # Calculate metrics
        correct = sum(1 for i, pred in enumerate(predictions) 
                     if pred["predicted"] == (AI_TEST_SAMPLES[i]["label"] == "ai"))
        accuracy = correct / len(AI_TEST_SAMPLES)
        avg_time = np.mean(inference_times) * 1000
        
        ai_results.append({
            "model": model_info["name"],
            "model_id": model_info["model_id"],
            "accuracy": accuracy,
            "avg_inference_ms": avg_time,
            "predictions": predictions
        })
        
        print(f"  Accuracy: {accuracy:.2%}")
        print(f"  Avg inference: {avg_time:.1f}ms")
        
    except Exception as e:
        print(f"  Failed: {e}")

print(f"\nBenchmark complete! {len(ai_results)} models tested.")

In [ ]:
# Per-category analysis
if ai_results:
    categories = {}
    for sample in AI_TEST_SAMPLES:
        cat = sample["source"]
        if cat not in categories:
            categories[cat] = []
        categories[cat].append(sample["label"])
    
    print("=== CATEGORY BREAKDOWN ===")
    for cat, labels in categories.items():
        ai_count = sum(1 for l in labels if l == "ai")
        human_count = len(labels) - ai_count
        print(f"{cat}: {ai_count} AI, {human_count} Human")

In [ ]:
# Threshold optimization curve simulation
# In production, use real validation data to find optimal confidence thresholds

thresholds = np.arange(0.3, 0.9, 0.05)
precision_scores = []
recall_scores = []
f1_scores = []

# Simulated data (replace with real validation results)
for threshold in thresholds:
    # Simulate precision decreasing with higher thresholds
    precision = 0.9 - (threshold - 0.5) * 0.5
    # Simulate recall increasing with lower thresholds  
    recall = 0.7 + (0.9 - threshold) * 0.4
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    precision_scores.append(max(0, min(1, precision)))
    recall_scores.append(max(0, min(1, recall)))
    f1_scores.append(max(0, min(1, f1)))

plt.figure(figsize=(10, 6))
plt.plot(thresholds, precision_scores, 'b-', label='Precision')
plt.plot(thresholds, recall_scores, 'r-', label='Recall')
plt.plot(thresholds, f1_scores, 'g-', label='F1 Score')
plt.xlabel('Confidence Threshold')
plt.ylabel('Score')
plt.title('Threshold Optimization Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

optimal_threshold = thresholds[np.argmax(f1_scores)]
print(f"Optimal threshold: {optimal_threshold:.2f} (F1: {max(f1_scores):.3f})")

In [ ]:
# Fine-tuning pipeline template
print("=== MODEL FINE-TUNING PIPELINE ===")
print("")
print("Steps to fine-tune a model for your specific use case:")
print("")
print("1. Prepare your dataset:")
print("   - Collect labeled examples (human-written vs AI-generated)")
print("   - Format as CSV/JSON with 'text' and 'label' columns")
print("   - Split into train/validation/test sets (80/10/10)")
print("")
print("2. Load base model and tokenizer:")
print("   model_name = 'bert-base-uncased'")
print("   tokenizer = AutoTokenizer.from_pretrained(model_name)")
print("   model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)")
print("")
print("3. Tokenize dataset:")
print("   def tokenize_function(examples):")
print("       return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=512)")
print("   tokenized_datasets = dataset.map(tokenize_function, batched=True)")
print("")
print("4. Set up training arguments:")
print("   training_args = TrainingArguments(")
print("       output_dir='./results',")
print("       num_train_epochs=3,")
print("       per_device_train_batch_size=16,")
print("       evaluation_strategy='epoch',")
print("       save_strategy='epoch',")
print("   )")
print("")
print("5. Initialize Trainer and train:")
print("   trainer = Trainer(")
print("       model=model,")
print("       args=training_args,")
print("       train_dataset=tokenized_datasets['train'],")
print("       eval_dataset=tokenized_datasets['validation'],")
print("   )")
print("   trainer.train()")
print("")
print("6. Evaluate and save:")
print("   metrics = trainer.evaluate(tokenized_datasets['test'])")
print("   trainer.save_model('./fine-tuned-model')")
print("   tokenizer.save_pretrained('./fine-tuned-model')")
print("")
print("7. Push to Hugging Face (optional):")
print("   model.push_to_hub('your-username/your-model-name')")
print("   tokenizer.push_to_hub('your-username/your-model-name')")